# Recursion for Data Engineering

This notebook covers:
- What recursion is
- Why recursion is useful
- When to use recursion
- Base case and recursive case
- Call stack
- Recursion vs iteration
- Nested JSON traversal
- JSON flattening
- Folder and data lake traversal
- Pipeline dependency traversal
- Category tree traversal
- Recursive retry simulation
- Practice problems
- Interview questions


## 1. What is Recursion?

Recursion is a programming technique where a function calls itself to solve a smaller version of the same problem.

A recursive function usually has two parts:

1. Base case: the condition where recursion stops.
2. Recursive case: the function calls itself with a smaller or simpler input.


In [ ]:
def countdown(n):
    if n == 0:
        print("Done")
        return

    print(n)
    countdown(n - 1)

countdown(5)


5
4
3
2
1
Done


## 2. Base Case and Recursive Case

The base case prevents infinite recursion.

The recursive case moves the input closer to the base case.


In [ ]:
def print_numbers(n):
    if n == 0:          # base case
        return

    print(n)            # current work
    print_numbers(n-1)  # recursive case

print_numbers(3)


3
2
1


## 3. Why Use Recursion?

Recursion is useful when a problem naturally contains smaller versions of itself.

Common data engineering examples:
- nested JSON from APIs
- nested folders in a data lake
- parent-child hierarchies
- product category trees
- pipeline dependency trees
- recursive search inside nested data


## 4. When to Use Recursion

Use recursion when:
- the data is nested
- the depth is unknown
- the structure is tree-like
- each level follows the same pattern
- solving one part requires solving smaller parts


## 5. When Not to Use Recursion

Avoid recursion when:
- a simple loop is easier
- the input is very deep
- memory usage is a concern
- the problem is naturally linear
- recursion makes the logic harder to understand

Python has a recursion limit.


In [ ]:
import sys
print(sys.getrecursionlimit())


1000


## 6. Call Stack

Every recursive call is stored on the call stack.

When the base case is reached, Python returns back from the deepest call to the first call.


In [ ]:
def show_stack(n):
    print("Entering:", n)

    if n == 0:
        print("Base case reached")
        return

    show_stack(n - 1)

    print("Returning from:", n)

show_stack(3)


Entering: 3
Entering: 2
Entering: 1
Entering: 0
Base case reached
Returning from: 1
Returning from: 2
Returning from: 3


## 7. Recursion vs Iteration

Many problems can be solved using both loops and recursion.


In [ ]:
def sum_iterative(numbers):
    total = 0
    for number in numbers:
        total += number
    return total

print(sum_iterative([10, 20, 30, 40]))


100


In [ ]:
def sum_recursive(numbers):
    if len(numbers) == 0:
        return 0

    return numbers[0] + sum_recursive(numbers[1:])

print(sum_recursive([10, 20, 30, 40]))


100


## 8. Factorial Example

Factorial is a classic recursive problem.

5 factorial = 5 * 4 * 3 * 2 * 1


In [ ]:
def factorial(n):
    if n == 0 or n == 1:
        return 1

    return n * factorial(n - 1)

print(factorial(5))


120


## 9. Data Engineering Use Case: Nested JSON

APIs often return nested JSON.

A recursive function can process nested dictionaries without knowing how deep they are.


In [ ]:
api_response = {
    "order_id": 1001,
    "customer": {
        "customer_id": 501,
        "name": "Riya",
        "location": {
            "city": "Delhi",
            "country": "India"
        }
    },
    "payment": {
        "method": "card",
        "status": "success"
    }
}

print(api_response)


{'order_id': 1001, 'customer': {'customer_id': 501, 'name': 'Riya', 'location': {'city': 'Delhi', 'country': 'India'}}, 'payment': {'method': 'card', 'status': 'success'}}


In [ ]:
a = [1,2,4]
b = [5,6,7]

b.extend(a)
print(a)

[1, 2, 4]


## 10. Extract All Keys from Nested JSON


In [ ]:
api_response = {
    "order_id": 1001,
    "customer": {
        "customer_id": 501,
        "name": "Riya",
        "location": {
            "city": "Delhi",
            "country": "India"
        }
    },
    "payment": {
        "method": "card",
        "status": "success"
    }
}


def extract_keys(data):
    keys = []

    if isinstance(data, dict):
        for key, value in data.items():
            keys.append(key)
            keys.extend(extract_keys(value))

    return keys

all_keys = extract_keys(api_response)
print(all_keys)


['order_id', 'customer', 'customer_id', 'name', 'location', 'city', 'country', 'payment', 'method', 'status']


In [ ]:
api_response -> keys = ["order_id","customer", ]
1001 -> keys =[]

{
        "customer_id": 501,
        "name": "Riya",
        "location": {
            "city": "Delhi",
            "country": "India"
        }
    } -> keys = ["customer_id", "name", "location", "city", "country", "payment","method","status"]

    {
            "city": "Delhi",
            "country": "India"
        } -> keys =

## 11. Flatten Nested JSON

Flattening nested JSON is useful before loading data into tabular systems.


In [ ]:
def flatten_json(data, parent_key="", separator="."):
    flattened = {}

    for key, value in data.items():
        new_key = parent_key + separator + key if parent_key else key

        if isinstance(value, dict):
            flattened.update(flatten_json(value, new_key, separator))
        else:
            flattened[new_key] = value

    return flattened

flat_record = flatten_json(api_response)
print(flat_record)


{'order_id': 1001, 'customer.customer_id': 501, 'customer.name': 'Riya', 'customer.location.city': 'Delhi', 'customer.location.country': 'India', 'payment.method': 'card', 'payment.status': 'success'}


## 12. Flatten Product Catalog Record


In [ ]:
product_record = {
    "product_id": 101,
    "name": "Laptop",
    "pricing": {
        "base_price": 80000,
        "discount": {
            "percentage": 10,
            "active": True
        }
    },
    "inventory": {
        "warehouse": {
            "city": "Bangalore",
            "stock": 25
        }
    }
}

flat_product = flatten_json(product_record)
print(flat_product)


{'product_id': 101, 'name': 'Laptop', 'pricing.base_price': 80000, 'pricing.discount.percentage': 10, 'pricing.discount.active': True, 'inventory.warehouse.city': 'Bangalore', 'inventory.warehouse.stock': 25}


In [ ]:
isinstance(product_record['inventory'], dict)

True

## 13. Search for a Key in Nested JSON


In [ ]:
def find_value_by_key(data, target_key):
    if isinstance(data, dict):
        for key, value in data.items():
            if key == target_key:
                return value

            result = find_value_by_key(value, target_key)
            if result is not None:
                return result

    return None

print(find_value_by_key(api_response, "city"))
print(find_value_by_key(api_response, "status"))
print(find_value_by_key(api_response, "missing_key"))


Delhi
success
None


## 14. Nested JSON with Lists

Real API data often contains lists inside dictionaries.


In [ ]:
order_payload = {
    "order_id": 1001,
    "items": [
        {
            "product_id": 201,
            "quantity": 2,
            "pricing": {
                "unit_price": 500,
                "currency": "INR"
            }
        },
        {
            "product_id": 202,
            "quantity": 1,
            "pricing": {
                "unit_price": 1200,
                "currency": "INR"
            }
        }
    ]
}

print(order_payload)


{'order_id': 1001, 'items': [{'product_id': 201, 'quantity': 2, 'pricing': {'unit_price': 500, 'currency': 'INR'}}, {'product_id': 202, 'quantity': 1, 'pricing': {'unit_price': 1200, 'currency': 'INR'}}]}


## 15. Extract All Values from Dicts and Lists


In [ ]:
def extract_all_values(data):
    values = []

    if isinstance(data, dict):
        for value in data.values():
            values.extend(extract_all_values(value))

    elif isinstance(data, list):
        for item in data:
            values.extend(extract_all_values(item))

    else:
        values.append(data)

    return values

print(extract_all_values(order_payload))


[1001, 201, 2, 500, 'INR', 202, 1, 1200, 'INR']


## 16. Flatten Any JSON with Dicts and Lists

List indexes are added into the flattened key path.


In [ ]:
def flatten_any_json(data, parent_key="", separator="."):
    flattened = {}

    if isinstance(data, dict):
        for key, value in data.items():
            new_key = parent_key + separator + key if parent_key else key
            flattened.update(flatten_any_json(value, new_key, separator))

    elif isinstance(data, list):
        for index, item in enumerate(data):
            new_key = parent_key + separator + str(index) if parent_key else str(index)
            flattened.update(flatten_any_json(item, new_key, separator))

    else:
        flattened[parent_key] = data

    return flattened

flat_order = flatten_any_json(order_payload)
print(flat_order)


{'order_id': 1001, 'items.0.product_id': 201, 'items.0.quantity': 2, 'items.0.pricing.unit_price': 500, 'items.0.pricing.currency': 'INR', 'items.1.product_id': 202, 'items.1.quantity': 1, 'items.1.pricing.unit_price': 1200, 'items.1.pricing.currency': 'INR'}


## 17. Data Engineering Use Case: Data Lake Folder Traversal

Data lakes often use nested folder structures.

Example:
raw/orders/2026/01/file.csv


In [ ]:
data_lake = {
    "raw": {
        "orders": {
            "2026": {
                "01": ["orders_01.csv", "orders_02.csv"],
                "02": ["orders_03.csv"]
            }
        },
        "customers": {
            "2026": {
                "01": ["customers_01.csv"]
            }
        }
    }
}

print(data_lake)


{'raw': {'orders': {'2026': {'01': ['orders_01.csv', 'orders_02.csv'], '02': ['orders_03.csv']}}, 'customers': {'2026': {'01': ['customers_01.csv']}}}}


## 18. Collect Files Recursively


In [ ]:
def collect_files(folder):
    files = []

    if isinstance(folder, dict):
        for value in folder.values():
            files.extend(collect_files(value))

    elif isinstance(folder, list):
        for item in folder:
            files.append(item)

    return files

all_files = collect_files(data_lake)
print(all_files)


['orders_01.csv', 'orders_02.csv', 'orders_03.csv', 'customers_01.csv']


## 19. Collect Files with Full Path


In [ ]:
def collect_files_with_path(folder, current_path=""):
    files = []

    if isinstance(folder, dict):
        for name, value in folder.items():
            new_path = current_path + "/" + name if current_path else name
            files.extend(collect_files_with_path(value, new_path))

    elif isinstance(folder, list):
        for file_name in folder:
            files.append(current_path + "/" + file_name)

    return files

file_paths = collect_files_with_path(data_lake)
print(file_paths)


['raw/orders/2026/01/orders_01.csv', 'raw/orders/2026/01/orders_02.csv', 'raw/orders/2026/02/orders_03.csv', 'raw/customers/2026/01/customers_01.csv']


## 20. Data Engineering Use Case: Pipeline Dependency Tree

Some data pipelines have dependency trees.

A report should run only after its dependent tables are ready.


In [ ]:
pipeline_dependencies = {
    "gold_sales_report": {
        "silver_orders": {
            "bronze_orders": {}
        },
        "silver_customers": {
            "bronze_customers": {}
        }
    }
}

print(pipeline_dependencies)


{'gold_sales_report': {'silver_orders': {'bronze_orders': {}}, 'silver_customers': {'bronze_customers': {}}}}


## 21. Execute Dependencies Recursively


In [ ]:
def execute_dependencies(task_tree, level=0):
    for task, dependencies in task_tree.items():
        execute_dependencies(dependencies, level + 1)
        print("  " * level + "Executing: " + task)

execute_dependencies(pipeline_dependencies)


    Executing: bronze_orders
  Executing: silver_orders
    Executing: bronze_customers
  Executing: silver_customers
Executing: gold_sales_report


## 22. Parent-Child Hierarchy Traversal

Hierarchical data appears in organizations, categories, locations, and reporting structures.


In [ ]:
org_chart = {
    "CEO": {
        "VP Engineering": {
            "Data Engineering Manager": {
                "Data Engineer 1": {},
                "Data Engineer 2": {}
            }
        },
        "VP Sales": {
            "Sales Manager": {
                "Sales Executive 1": {}
            }
        }
    }
}

def print_hierarchy(tree, level=0):
    for person, reports in tree.items():
        print("  " * level + person)
        print_hierarchy(reports, level + 1)

print_hierarchy(org_chart)


CEO
  VP Engineering
    Data Engineering Manager
      Data Engineer 1
      Data Engineer 2
  VP Sales
    Sales Manager
      Sales Executive 1


## 23. Count Nodes in a Hierarchy


In [ ]:
def count_nodes(tree):
    count = 0

    for node, children in tree.items():
        count += 1
        count += count_nodes(children)

    return count

print(count_nodes(org_chart))


8


## 24. Product Category Tree


In [ ]:
category_tree = {
    "Electronics": {
        "Mobiles": {
            "Android": {},
            "iOS": {}
        },
        "Laptops": {
            "Gaming": {},
            "Business": {}
        }
    },
    "Fashion": {
        "Men": {},
        "Women": {}
    }
}

print_hierarchy(category_tree)


Electronics
  Mobiles
    Android
    iOS
  Laptops
    Gaming
    Business
Fashion
  Men
  Women


## 25. Collect Leaf Categories

Leaf categories are categories with no child categories.


In [ ]:
def collect_leaf_nodes(tree):
    leaves = []

    for node, children in tree.items():
        if len(children) == 0:
            leaves.append(node)
        else:
            leaves.extend(collect_leaf_nodes(children))

    return leaves

print(collect_leaf_nodes(category_tree))


['Android', 'iOS', 'Gaming', 'Business', 'Men', 'Women']


## 26. Recursive Retry Simulation

Retry logic is common when APIs fail.

This is a simplified recursive retry example.


In [ ]:
def retry_api_call(attempt, max_attempts):
    print("Attempt:", attempt)

    if attempt == max_attempts:
        return "Failed after max retries"

    if attempt == 3:
        return "Success"

    return retry_api_call(attempt + 1, max_attempts)

result = retry_api_call(1, 5)
print(result)


Attempt: 1
Attempt: 2
Attempt: 3
Success


## 27. Recursion Depth Risk

Deep recursion can raise a RecursionError.

The following code is commented intentionally.


In [ ]:
# def infinite_recursion():
#     return infinite_recursion()
#
# infinite_recursion()


## 28. Safe Recursion with Input Validation


In [ ]:
def safe_countdown(n):
    if n < 0:
        return "Invalid input"

    if n == 0:
        return "Done"

    print(n)
    return safe_countdown(n - 1)

print(safe_countdown(5))
print(safe_countdown(-1))


5
4
3
2
1
Done
Invalid input


# Practice Problems


## 29. Practice Problem 1: Flatten Nested Customer JSON


In [ ]:
customer_record = {
    "customer_id": 501,
    "profile": {
        "name": "Riya",
        "contact": {
            "email": "riya@example.com",
            "phone": "9999999999"
        }
    },
    "address": {
        "city": "Delhi",
        "country": "India"
    }
}

# Write solution here


## 30. Practice Problem 2: Collect Files from Nested Data Lake Structure


In [ ]:
storage = {
    "landing": {
        "orders": ["orders_day1.csv", "orders_day2.csv"],
        "users": ["users_day1.csv"]
    },
    "processed": {
        "orders": {
            "2026": ["orders_clean.csv"]
        }
    }
}

# Write solution here


## 31. Practice Problem 3: Search for a Key in Nested API Response


In [ ]:
api_data = {
    "response": {
        "metadata": {
            "request_id": "abc123",
            "source": "mobile_app"
        },
        "data": {
            "order": {
                "order_id": 1001,
                "status": "completed"
            }
        }
    }
}

target_key = "status"

# Write solution here


## 32. Practice Problem 4: Count Tasks in Dependency Tree


In [ ]:
tasks = {
    "final_report": {
        "sales_summary": {
            "sales_raw": {}
        },
        "customer_summary": {
            "customers_raw": {}
        },
        "product_summary": {
            "products_raw": {}
        }
    }
}

# Write solution here


## 33. Practice Problem 5: Collect Leaf Categories


In [ ]:
catalog_tree = {
    "Home": {
        "Furniture": {
            "Tables": {},
            "Chairs": {}
        },
        "Decor": {
            "Lamps": {},
            "Paintings": {}
        }
    },
    "Grocery": {
        "Snacks": {},
        "Beverages": {}
    }
}

# Write solution here


# Interview Questions


## 34. What is Recursion?

Recursion is a technique where a function calls itself to solve a smaller version of the same problem.

A recursive function requires:
- base case
- recursive case


## 35. When Should Recursion Be Used?

Recursion should be used when the problem has a naturally nested or hierarchical structure.

Examples:
- folder traversal
- nested JSON flattening
- tree traversal
- dependency resolution
- parent-child hierarchy traversal


## 36. Why Use Recursion?

Recursion can make code simpler and more natural for problems that are recursive in structure.

It avoids manually managing many levels of nested loops when the depth is unknown.


## 37. What Happens If There Is No Base Case?

Without a base case, the function keeps calling itself.

Eventually Python raises a RecursionError because the call stack limit is reached.


## 38. What Is the Difference Between Recursion and Iteration?

Recursion:
- function calls itself
- useful for nested or tree-like data
- uses call stack

Iteration:
- uses loops
- useful for linear processing
- usually more memory efficient


## 39. Why Is Recursion Useful in Data Engineering?

Data engineering often deals with nested or hierarchical data.

Examples:
- nested API JSON
- nested folders in cloud storage
- organization hierarchies
- product category trees
- pipeline dependency graphs

Recursion helps process these structures when their depth is unknown.


## 40. Summary

Recursion:
- solves a problem by calling the same function on smaller inputs
- requires a base case
- requires a recursive case
- is useful for nested data and hierarchical structures

Data engineering use cases:
- flatten nested JSON
- collect files from nested folders
- traverse dependency trees
- process category hierarchies
- search nested API responses


In [ ]:
{
    "product_id": 1,
    "name": "Smartphone",
    "description": "High-end smartphone with advanced features.",
    "price": 599.99,
    "unit": "Piece",
    "image": "https://example.com/images/smartphone.jpg",
    "discount": 10,
    "availability": True,
    "brand": "BrandX",
    "category": "Electronics",
    "rating": 4.5,
    "reviews": [
      {
        "user_id": 1,
        "rating": 5,
        "comment": "Great phone with a superb camera!"
      },
      {
        "user_id": 2,
        "rating": 4,
        "comment": "Good performance, but the battery life could be better."
      }
    ]
  }

{'product_id': 1,
 'name': 'Smartphone',
 'description': 'High-end smartphone with advanced features.',
 'price': 599.99,
 'unit': 'Piece',
 'image': 'https://example.com/images/smartphone.jpg',
 'discount': 10,
 'availability': True,
 'brand': 'BrandX',
 'category': 'Electronics',
 'rating': 4.5,
 'reviews': [{'user_id': 1,
   'rating': 5,
   'comment': 'Great phone with a superb camera!'},
  {'user_id': 2,
   'rating': 4,
   'comment': 'Good performance, but the battery life could be better.'}]}

In [ ]:
{
    product_id,
    name,
}